In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score

import joblib

In [3]:
df = pd.read_csv("../../TestDataSet/categorization/transaction_category_dataset.csv")

In [4]:
df['category'].value_counts()

category
Cash Withdrawal    3600
Entertainment      3600
Rent               3600
Health             3600
Tax                3600
Investment         3600
Fuel               3600
Loan EMI           3600
Electronics        3600
Shopping           3600
Utilities          3596
Insurance          3591
Bank Charges       3588
Income             3588
Education          3588
Transport          3588
Travel             3588
Interest Income    3587
Transfer           3587
Subscriptions      3586
Groceries          3586
Personal Care      3586
Food & Dining      3570
Name: count, dtype: int64

In [5]:
df["category"] = df["category"].str.strip()

In [6]:
category_mapping = {
    "food": "Food & Dining",
    "transport": "Transport",
    "shopping": "Shopping",
    "entertainment": "Entertainment",
    "technology": "Electronics"
}

df["category"] = df["category"].replace(category_mapping)

In [8]:
x = df["narration"]
y=df['category']

In [9]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.2, random_state = 42)


In [10]:
model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1,2),
            min_df=2
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000
        )
    )
])

In [11]:
model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](23,)","['Bank Charges','Cash Withdrawal','Education',...,'Transport','Travel', 'Utilities']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [12]:
predictions = model.predict(x_test)

In [13]:
accuracy = accuracy_score(y_test, predictions)
print(accuracy)

0.9961878252450683


In [14]:
test_merchants = [
    "Dominos",
    "Pizza Hut",
    "McDonalds",
    "KFC",
    "Starbucks",
    "IRCTC",
    "BookMyShow"
]

for m in test_merchants:
    print(m, "->", model.predict([m])[0])

Dominos -> Food & Dining
Pizza Hut -> Food & Dining
McDonalds -> Food & Dining
KFC -> Food & Dining
Starbucks -> Food & Dining
IRCTC -> Travel
BookMyShow -> Entertainment


In [15]:
import joblib
joblib.dump(model, "../ml/model.pkl")

['../ml/model.pkl']

In [16]:
print(len(df))

82629


In [21]:
print(df["category"].nunique())

23


In [18]:
print(df.columns)


Index(['narration', 'category'], dtype='str')


In [19]:
print(df.head())


                                   narration         category
0                            MANDATE-ATM BOB  Cash Withdrawal
1      IMPS/331463695507/CHEQUE BOOK CHARGES     Bank Charges
2                 NEFT/840880734956/SONY LIV    Entertainment
3  UPI/OUT/751241994716/housr@okaxis/Payment             Rent
4  UPI/OUT/962963962395/metropol@ybl/Payment           Health


In [24]:
tests = [
    "ordered pizza for dinner",
    "flight ticket booking",
    "movie ticket booking",
    "doctor consultation",
    "monthly netflix subscription",
    "electricity bill payment",
    "haircut at salon"
]

for t in tests:
    print(t, "->", model.predict([t])[0])

ordered pizza for dinner -> Food & Dining
flight ticket booking -> Travel
movie ticket booking -> Travel
doctor consultation -> Food & Dining
monthly netflix subscription -> Entertainment
electricity bill payment -> Utilities
haircut at salon -> Personal Care


In [29]:
food_rows = df[df["category"] == "Food & Dining"]

for desc in food_rows["narration"].sample(30):
    print(desc)

IMPS/866098217330/KFC
UPI/OUT/899702427784/motimaha@oksbi/Payment
POS PURCHASE INNERCHEF
UPI/OUT/895175666369/freshmen@ibl/Payment
IMPS/776049775751/BURGER KING
BILLDESK-BIKANERVALA-963491
MANDATE-MOTI MAHAL
BILLDESK-THEOBROMA-848901
IMPS/951877531146/NANDOS
UPI/ZOMATO/055819015621
UPI/UDUPI PALACE/520540419938
UPI/DR/803338451490/HALDIRAMS
BILLDESK-KWALITY WALLS-694983
POS 696082 ZOMATO
UPI/OUT/878733896146/sagarrat@okaxis/Payment
UPI/FAASOS
ECS-SARAVANA BHAVAN
NEFT-BIKANERVALA-22833184
UPI/OUT/996711558179/behrouzb@kotak/Payment
UPI PAYMENT KFC
AUTO DEBIT ROLLS MANIA
BILLDESK-ROLLS MANIA-375199
UPI/OUT/532540948934/box8@axl/Payment
POS 682990 CHAI POINT
NEFT-PARADISE BIRYANI-00975626
MANDATE-BARBEQUE NATION
MANDATE-MOTI MAHAL
POS 682990 CHAI POINT
POS/011861/BIKANERVALA
ACH DR CHAAYOS


In [30]:
tests = [
    "zomato payment",
    "swiggy instamart",
    "dominos order",
    "biryani order",
    "ordered pizza for dinner",
    "restaurant dinner"
]

for t in tests:
    print(t, "->", model.predict([t])[0])

zomato payment -> Food & Dining
swiggy instamart -> Groceries
dominos order -> Food & Dining
biryani order -> Food & Dining
ordered pizza for dinner -> Food & Dining
restaurant dinner -> Food & Dining


In [31]:
tests = [
    "burger meal",
    "coffee at cafe",
    "annual health checkup",
    "bus ticket booking",
    "amazon electronics purchase",
    "hair spa treatment",
    "netflix premium plan",
    "monthly rent payment"
]

for t in tests:
    print(t, "->", model.predict([t])[0])
    

burger meal -> Food & Dining
coffee at cafe -> Food & Dining
annual health checkup -> Insurance
bus ticket booking -> Travel
amazon electronics purchase -> Shopping
hair spa treatment -> Food & Dining
netflix premium plan -> Subscriptions
monthly rent payment -> Rent
